# CAPSTONE 24 — Facial Expression Classification: Dataset Bias Study

**Research question:** How do dataset composition and image conditions affect a classifier's performance, and why should results be interpreted cautiously?

**Interpretation boundary:** Labels represent dataset annotations of visible expression categories, not verified inner emotional states. This notebook is an educational audit, not a real-world emotion detector.

Run from the project root after creating `data/manifest.csv`. Do not upload or publish dataset images. Verify your dataset terms first.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageEnhance, ImageFilter
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score
from skimage.feature import hog

SEED = 42
PROJECT_ROOT = Path.cwd()
MANIFEST = PROJECT_ROOT / "data" / "manifest.csv"
IMAGE_SIZE = (48, 48)


## 1. Dataset documentation review (complete before modelling)

Record the official dataset URL, version/date accessed, permitted use, consent/access conditions, label definitions, known limitations, and whether your planned storage/visualization is allowed. Do not continue unless the terms permit your educational use.

In [ ]:
dataset_review = {
    "dataset_name": "Extended Cohn-Kanade (CK+), subject to official access approval",
    "official_documentation_url": "https://ckplus.jeffcohn.net/request",
    "access_and_use_terms_reviewed": False,  # change only after personally reviewing current terms
    "faculty_approval_obtained_if_required": False,
    "consent_and_publication_terms_reviewed": False,
    "label_definitions_reviewed": False,
    "redistribution_prohibited_or_limited": True,
}
print(pd.Series(dataset_review))
if not all(dataset_review[k] for k in [
    "access_and_use_terms_reviewed",
    "faculty_approval_obtained_if_required",
    "consent_and_publication_terms_reviewed",
    "label_definitions_reviewed"
]):
    print("\nSTOP CHECK: review the official terms and complete this checklist before using real data.")


## 2. Load the manifest and audit data quality

`manifest.csv` must contain `image_path`, `label`, and `subject_id`. `condition` and `subgroup` are optional documented metadata fields. Leave unavailable metadata blank; never infer demographic traits from images.

In [ ]:
if not MANIFEST.exists():
    raise FileNotFoundError(f"Create {MANIFEST} using data/manifest_template.csv, then rerun.")
df = pd.read_csv(MANIFEST).fillna("")
required = {"image_path", "label", "subject_id"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Manifest is missing columns: {sorted(missing)}")
df = df[(df["image_path"].astype(str).str.len() > 0) &
        (df["label"].astype(str).str.len() > 0) &
        (df["subject_id"].astype(str).str.len() > 0)].copy()
df["resolved_path"] = df["image_path"].apply(lambda p: str((PROJECT_ROOT / p).resolve()) if not Path(p).is_absolute() else p)
df["exists"] = df["resolved_path"].apply(lambda p: Path(p).is_file())
print("Rows in usable manifest:", len(df))
print("Missing image files:", int((~df["exists"]).sum()))
if (~df["exists"]).any():
    display(df.loc[~df["exists"], ["image_path", "label", "subject_id"]].head(10))
df = df[df["exists"]].reset_index(drop=True)
print("Unique subjects:", df["subject_id"].nunique())
print("Classes:", df["label"].nunique())


In [ ]:
class_counts = df["label"].value_counts().sort_index()
display(class_counts.rename("image_count").to_frame())
ax = class_counts.plot(kind="bar", title="Class balance (number of images)")
ax.set_xlabel("Dataset label"); ax.set_ylabel("Images"); plt.tight_layout(); plt.show()

print("Images per subject — summary")
display(df.groupby("subject_id").size().describe().to_frame("images_per_subject"))
if "condition" in df.columns and (df["condition"].astype(str).str.strip() != "").any():
    display(pd.crosstab(df["condition"], df["label"], margins=True))
else:
    print("No documented image-condition metadata supplied; subgroup comparison by condition is unavailable.")
if "subgroup" in df.columns and (df["subgroup"].astype(str).str.strip() != "").any():
    display(pd.crosstab(df["subgroup"], df["label"], margins=True))
else:
    print("No documented subgroup metadata supplied; demographic subgroup comparison is unavailable.")


## 3. Load images and extract HOG features

A simple linear SVM is used as a transparent baseline. HOG features summarize local edge/shape patterns; this is not a claim that facial appearance reveals internal emotion.

In [ ]:
def load_gray(path, size=IMAGE_SIZE):
    with Image.open(path) as im:
        im = im.convert("L").resize(size)
        return np.asarray(im, dtype=np.uint8)

images, valid_rows = [], []
for _, row in df.iterrows():
    try:
        images.append(load_gray(row["resolved_path"]))
        valid_rows.append(row)
    except Exception as e:
        warnings.warn(f"Could not read {row['image_path']}: {e}")

df = pd.DataFrame(valid_rows).reset_index(drop=True)
X_img = np.stack(images)
y = df["label"].astype(str).to_numpy()
groups = df["subject_id"].astype(str).to_numpy()

def hog_features(arrays):
    return np.asarray([hog(im, orientations=9, pixels_per_cell=(8,8),
                           cells_per_block=(2,2), block_norm="L2-Hys")
                       for im in arrays])

X = hog_features(X_img)
print("Feature matrix:", X.shape)


## 4. Leakage-aware split by identity

Images from one subject must not appear in both train and test. A grouped split helps prevent the model from benefiting from subject-specific cues. This is still only one split; repeat with grouped cross-validation if time permits.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]
assert set(groups_train).isdisjoint(set(groups_test)), "Identity leakage detected!"
print("Train images:", len(train_idx), "Test images:", len(test_idx))
print("Train subjects:", len(set(groups_train)), "Test subjects:", len(set(groups_test)))
print("Classes absent from train:", sorted(set(y_test) - set(y_train)))


In [ ]:
model = Pipeline([
    ("scale", StandardScaler()),
    ("classifier", LinearSVC(class_weight="balanced", random_state=SEED, max_iter=10000))
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("Balanced accuracy:", round(balanced_accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, zero_division=0))
labels = sorted(set(y_train) | set(y_test))
cm = confusion_matrix(y_test, pred, labels=labels)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(xticks_rotation=45)
plt.title("Held-out subject-grouped confusion matrix")
plt.tight_layout(); plt.show()


## 5. Class-wise error audit

Look for systematic confusion between visually similar or ambiguous categories. Do not interpret a predicted label as evidence of how a person actually feels.

In [ ]:
report = classification_report(y_test, pred, output_dict=True, zero_division=0)
class_report = pd.DataFrame(report).T
display(class_report.loc[[c for c in labels if c in class_report.index],
                         ["precision", "recall", "f1-score", "support"]])
errors = np.where(pred != y_test)[0]
error_rows = pd.DataFrame({
    "manifest_row": test_idx[errors],
    "true_dataset_label": y_test[errors],
    "predicted_dataset_label": pred[errors],
})
print("Misclassified test images:", len(errors), "of", len(y_test))
display(error_rows.head(20))


## 6. Controlled image-condition sensitivity test

The following perturbations are synthetic stress tests applied to held-out images: reduced brightness, contrast, and mild blur. They are not a substitute for real-world validation, and they do not represent all lighting/pose/quality conditions. The test images remain the same people as the held-out set, never training data.

In [ ]:
def perturb(im, mode):
    pil = Image.fromarray(im)
    if mode == "darker":
        pil = ImageEnhance.Brightness(pil).enhance(0.55)
    elif mode == "lower_contrast":
        pil = ImageEnhance.Contrast(pil).enhance(0.55)
    elif mode == "blur":
        pil = pil.filter(ImageFilter.GaussianBlur(radius=1.2))
    return np.asarray(pil, dtype=np.uint8)

condition_results = []
for condition in ["original", "darker", "lower_contrast", "blur"]:
    arr = X_img[test_idx] if condition == "original" else np.stack([perturb(im, condition) for im in X_img[test_idx]])
    feats = hog_features(arr)
    p = model.predict(feats)
    condition_results.append({
        "condition": condition,
        "accuracy": accuracy_score(y_test, p),
        "balanced_accuracy": balanced_accuracy_score(y_test, p),
        "n_test": len(y_test)
    })
condition_df = pd.DataFrame(condition_results)
display(condition_df)
ax = condition_df.plot(x="condition", y=["accuracy", "balanced_accuracy"], kind="bar",
                       title="Sensitivity to controlled image perturbations", ylim=(0,1))
ax.set_ylabel("Score"); ax.set_xlabel("Test condition"); plt.xticks(rotation=20); plt.tight_layout(); plt.show()


## 7. Subgroup / condition evaluation (only if documented metadata exists)

Only use metadata supplied by the dataset and permitted for this purpose. Do not label people by visual guesses. Report sample sizes and avoid strong conclusions from tiny groups.

In [ ]:
def subgroup_metrics(frame, truth, predicted, column, min_n=10):
    if column not in frame.columns:
        print(f"No {column!r} column.")
        return pd.DataFrame()
    values = frame[column].astype(str).str.strip()
    usable = values.ne("") & values.ne("nan")
    if not usable.any():
        print(f"No documented {column!r} metadata available.")
        return pd.DataFrame()
    tmp = pd.DataFrame({"group": values[usable].to_numpy(),
                        "truth": np.asarray(truth)[usable.to_numpy()],
                        "pred": np.asarray(predicted)[usable.to_numpy()]})
    rows = []
    for group, part in tmp.groupby("group"):
        rows.append({"group": group, "n": len(part),
                     "accuracy": accuracy_score(part.truth, part.pred),
                     "balanced_accuracy": balanced_accuracy_score(part.truth, part.pred)
                         if part.truth.nunique() > 1 else np.nan,
                     "small_sample_warning": len(part) < min_n})
    return pd.DataFrame(rows).sort_values("n", ascending=False)

test_frame = df.iloc[test_idx].reset_index(drop=True)
display(subgroup_metrics(test_frame, y_test, pred, "condition"))
display(subgroup_metrics(test_frame, y_test, pred, "subgroup"))


## 8. Private error gallery (do not publish face images)

Inspect a small number of errors locally only if dataset terms allow it. Never put identifiable face images in public reports or slides unless explicit permission permits this. The cell below shows a few images inside the current private notebook; skip it if the agreement disallows display.

In [ ]:
# Set SHOW_PRIVATE_ERROR_GALLERY = True only if the dataset agreement allows local display.
SHOW_PRIVATE_ERROR_GALLERY = False
if SHOW_PRIVATE_ERROR_GALLERY and len(errors):
    n_show = min(8, len(errors))
    fig, axes = plt.subplots(2, 4, figsize=(10, 5))
    for ax, eidx in zip(axes.flat, errors[:n_show]):
        im = X_img[test_idx[eidx]]
        ax.imshow(im, cmap="gray")
        ax.set_title(f"True: {y_test[eidx]}\nPred: {pred[eidx]}", fontsize=8)
        ax.axis("off")
    for ax in axes.flat[n_show:]:
        ax.axis("off")
    plt.suptitle("PRIVATE ERROR REVIEW — DO NOT EXPORT OR SHARE")
    plt.tight_layout(); plt.show()
else:
    print("Gallery disabled by default to reduce privacy and redistribution risks.")


## 9. Interpretation checklist and export

Answer these in your report:
1. Which classes have the lowest recall and why might labels or sample counts matter?
2. How does class balance affect accuracy versus balanced accuracy?
3. Does identity grouping change the interpretation compared with a random image split?
4. Which synthetic perturbation changes performance most? What does this not tell us?
5. Were subgroup/condition comparisons possible? If not, explicitly say metadata was unavailable.
6. What are the limits of the dataset (posed expressions, sample size, culture, label ambiguity, image conditions, consent/access restrictions)?
7. Why must the model not be used for surveillance or consequential decisions?

Save the real metrics from the notebook; do not fill report tables with made-up numbers.

In [ ]:
# Export summary metrics for the report (only after successful run)
out_dir = PROJECT_ROOT / "outputs"
out_dir.mkdir(exist_ok=True)
pd.DataFrame([{
    "accuracy": accuracy_score(y_test, pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, pred),
    "n_train": len(y_train),
    "n_test": len(y_test),
    "n_train_subjects": len(set(groups_train)),
    "n_test_subjects": len(set(groups_test)),
}]).to_csv(out_dir / "overall_metrics.csv", index=False)
class_report.to_csv(out_dir / "class_wise_report.csv")
condition_df.to_csv(out_dir / "condition_sensitivity.csv", index=False)
print("Saved outputs to", out_dir)
